# TransitPulse MBTA — benchmark trên Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nqMinh-GR/de_btl/blob/main/notebooks/TransitPulse_MBTA_Colab_Benchmark.ipynb)

Open in Colab: `https://colab.research.google.com/github/nqMinh-GR/de_btl/blob/main/notebooks/TransitPulse_MBTA_Colab_Benchmark.ipynb`

**Runtime → Run all.** Notebook này chỉ đo. Nó không mở UI demo.

Run All làm lần lượt: clone, bootstrap, bật Kafka và Cassandra, tắt API nếu còn sống, chạy benchmark, zip và tải `benchmarks.zip`.

Scenario luôn chạy, mỗi cái một lần: `stress_100`, `stress_500`, `stress_1000`, rồi bốn recovery (`recovery_partial_write`, `recovery_after_cas`, `recovery_cassandra_down`, `recovery_collector_outbox`). Stress là dữ liệu synthetic, không phải xe MBTA.

`replay_1x`, `replay_10x`, `replay_50x` chỉ được thêm khi đã có `recording_manifest.json` của một session live đã dừng, hoặc khi điền `RECORDING_MANIFEST` ở cell tham số.

Cả suite có thể chạy nhiều giờ. Chỉ dòng `MEASURED` là số đã đo. Latency API, latency UI và p95 ghi Cassandra không nằm trong report này.


In [ ]:
# [2/7] Tham số dùng chung với notebook demo (clone, bootstrap, Kafka, Cassandra).
# Notebook này không mở UI và không bật tunnel. Không đặt token vào notebook.
GITHUB_REPO_URL = "https://github.com/nqMinh-GR/de_btl.git"
GIT_REF = "main"              # Khi demo chính thức, điền commit SHA (40 ký tự hex) hoặc release tag.
REFRESH_REF = True            # Fetch và checkout lại GIT_REF ở cell clone (bật sau mỗi lần push).
DEMO_MODE = "live"            # synthetic | replay | live
TP_REPO_DIR = "/content/transitpulse-mbta"
TP_WORKDIR = "/content/transitpulse-runtime"

# 10 tuyến bus MBTA (route_type 3) có trong GTFS. Prepare chạy source-check và chỉ giữ tuyến hợp lệ.
ROUTE_IDS = ["1", "15", "22", "23", "28", "32", "39", "57", "66", "77"]
RECORDING_MANIFEST = ""      # Bắt buộc khi chọn replay.
REPLAY_SPEED = 10.0
SYNTHETIC_SEED = 42
DEMO_DURATION_SECONDS = 600  # Wall-time budget; không nhầm với recording/source duration.

RUN_UNIT_TESTS = True
RUN_GOLDEN_SMOKE = True
RUN_BENCHMARK = False
RUN_INTEGRATION_TESTS = False  # Tùy chọn: integration tests trên Kafka/Spark/Cassandra thật (lâu, tốn RAM).
EXPORT_ARTIFACTS = True
STOP_AFTER_RUN_ALL = False   # Giữ UI tương tác; cung cấp nút Stop riêng.
MOUNT_DRIVE = False          # Tùy chọn sao lưu ZIP, không làm working database.
STRICT_RUN_ALL = False       # True: cell 18 báo lỗi nếu còn bước FAIL. False: Run All chạy hết, lỗi vẫn hiện đỏ.

ENABLE_CLOUDFLARE_TUNNEL = False  # Benchmark không cần link công khai; bootstrap bỏ qua cloudflared.
TUNNEL_TARGET = "react"      # react | dashboard | api | both | all (api/both/all công khai cả API chỉ đọc)

# ---------------------------------------------------------------------------------------------------
# Phần dưới: kiểm tra tham số + hàm _ctx() dùng chung cho mọi cell (không cần sửa).
# Sau khi restart kernel: chạy lại cell này rồi chạy tiếp cell cần dùng (dịch vụ nền vẫn chạy).
import importlib.util
import json
import os
import re
import sys
from pathlib import Path
from types import SimpleNamespace

TP_PROFILE = 'colab-native'
_MODES = ('synthetic', 'replay', 'live')
_TUNNEL_TARGETS = ('dashboard', 'api', 'react', 'both', 'all')
_ROUTE_ID_RE = '[A-Za-z0-9._-]{1,64}'
_GIT_REF_RE = '[A-Za-z0-9][A-Za-z0-9._/-]{0,199}'


class ParameterError(ValueError):
    '''Tham số không hợp lệ: notebook dừng ngay, không tự đổi mode.'''


def _params():
    '''Đọc và kiểm tra tham số (gọi lại ở mọi cell nên giá trị papermill/sửa tay đều được kiểm tra).'''
    g = globals()
    errors, notes = [], []
    mode = str(g.get('DEMO_MODE', '')).strip().lower()
    if mode not in _MODES:
        errors.append('DEMO_MODE phải là một trong %s (đang là %r).' % (', '.join(_MODES), g.get('DEMO_MODE')))
    repo_url = str(g.get('GITHUB_REPO_URL', '')).strip()
    if not repo_url.startswith('https://'):
        errors.append('GITHUB_REPO_URL phải là URL https (không chứa token).')
    ref = str(g.get('GIT_REF', '')).strip()
    if not re.fullmatch(_GIT_REF_RE, ref) or '..' in ref:
        errors.append('GIT_REF không hợp lệ: %r.' % ref)
    repo_dir = str(g.get('TP_REPO_DIR', '')).strip()
    workdir = str(g.get('TP_WORKDIR', '')).strip()
    for name, value in (('TP_REPO_DIR', repo_dir), ('TP_WORKDIR', workdir)):
        if not os.path.isabs(value) or os.path.normpath(value) in ('/', '/content', '/content/drive'):
            errors.append('%s phải là đường dẫn tuyệt đối riêng (đang là %r).' % (name, value))
    if repo_dir and workdir and os.path.normpath(repo_dir) == os.path.normpath(workdir):
        errors.append('TP_REPO_DIR và TP_WORKDIR phải khác nhau.')
    if workdir.startswith('/content/drive'):
        errors.append('TP_WORKDIR không được nằm trên Google Drive (Kafka/Cassandra/checkpoint cần disk local).')
    routes = g.get('ROUTE_IDS') or []
    if isinstance(routes, str):
        routes = [r.strip() for r in routes.split(',') if r.strip()]
    if not isinstance(routes, (list, tuple)):
        errors.append('ROUTE_IDS phải là list các route_id.')
        routes = []
    routes = [str(r).strip() for r in routes]
    bad = [r for r in routes if not re.fullmatch(_ROUTE_ID_RE, r)]
    if bad:
        errors.append('route_id không hợp lệ: %s' % bad)
    if len(set(routes)) != len(routes):
        errors.append('ROUTE_IDS có phần tử trùng.')
    if len(routes) > 10:
        errors.append('Tối đa 10 route trong scope demo (đang có %d).' % len(routes))
    manifest = str(g.get('RECORDING_MANIFEST') or '').strip()
    if manifest and not os.path.isabs(manifest):
        manifest = os.path.join(repo_dir, manifest)
    try:
        speed = float(g.get('REPLAY_SPEED'))
    except (TypeError, ValueError):
        speed = float('nan')
    if not (0 < speed <= 100):
        errors.append('REPLAY_SPEED phải trong (0, 100] (đang là %r).' % g.get('REPLAY_SPEED'))
    seed = g.get('SYNTHETIC_SEED')
    if isinstance(seed, bool) or not isinstance(seed, int) or seed < 0:
        errors.append('SYNTHETIC_SEED phải là số nguyên >= 0.')
    duration = g.get('DEMO_DURATION_SECONDS')
    if isinstance(duration, bool) or not isinstance(duration, int) or not (60 <= duration <= 7200):
        errors.append('DEMO_DURATION_SECONDS phải là số nguyên trong [60, 7200].')
    flags = {}
    for name in ('RUN_UNIT_TESTS', 'RUN_GOLDEN_SMOKE', 'RUN_BENCHMARK', 'RUN_INTEGRATION_TESTS',
                 'EXPORT_ARTIFACTS', 'STOP_AFTER_RUN_ALL', 'MOUNT_DRIVE', 'STRICT_RUN_ALL',
                 'ENABLE_CLOUDFLARE_TUNNEL'):
        value = g.get(name, False)
        if not isinstance(value, bool):
            errors.append('%s phải là True/False.' % name)
        flags[name.lower()] = bool(value)
    tunnel_target = str(g.get('TUNNEL_TARGET', 'react')).strip().lower()
    if tunnel_target not in _TUNNEL_TARGETS:
        errors.append('TUNNEL_TARGET phải là một trong %s (đang là %r).' % (', '.join(_TUNNEL_TARGETS),
                                                                           g.get('TUNNEL_TARGET')))
    if mode == 'replay' and not manifest:
        errors.append('Mode replay cần RECORDING_MANIFEST = đường dẫn tới recording_manifest.json của một bundle '
                      'đã ghi (collector record, hoặc ZIP artifact đã giải nén vào TP_WORKDIR/data/inputs). '
                      'Notebook không tự sinh recording MBTA giả và không tự chuyển sang synthetic.')
    if mode == 'live' and not routes:
        errors.append('Mode live cần ROUTE_IDS (1-10 bus route đã kiểm tra bằng source-check). '
                      'Notebook không tự chọn tuyến và không chạy synthetic dưới nhãn LIVE.')
    if mode != 'replay' and manifest:
        notes.append('RECORDING_MANIFEST bị bỏ qua vì DEMO_MODE=%s.' % mode)
    if mode == 'synthetic' and routes:
        notes.append('ROUTE_IDS bị bỏ qua ở synthetic (fixture có 5 tuyến synthetic riêng).')
    if flags.get('enable_cloudflare_tunnel'):
        notes.append('Tunnel bật: link https://*.trycloudflare.com là CÔNG KHAI, ai có link đều xem được (%s).'
                     % tunnel_target)
    if errors:
        raise ParameterError('Tham số không hợp lệ:' + ''.join(chr(10) + ' - ' + e for e in errors))
    return SimpleNamespace(mode=mode, repo_url=repo_url, git_ref=ref, repo_dir=repo_dir, workdir=workdir,
                           route_ids=routes, manifest=manifest, speed=speed, seed=seed, duration=duration,
                           tunnel_target=tunnel_target, notes=notes, **flags)


def _load_ui(repo_dir=None):
    '''Nạp (lại) notebooks/colab_ui.py từ repo đã clone; an toàn sau restart kernel.'''
    path = Path(repo_dir or _params().repo_dir) / 'notebooks' / 'colab_ui.py'
    if not path.is_file():
        raise FileNotFoundError('Không thấy %s — chạy cell 3 (clone) trước.' % path)
    name = 'transitpulse_colab_ui'
    spec = importlib.util.spec_from_file_location(name, str(path))
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


def _ctx():
    '''Ngữ cảnh của mọi cell sau bootstrap: tham số đã kiểm tra, runtime.json, colab_ui, session gần nhất.'''
    p = _params()
    ui = _load_ui(p.repo_dir)
    rt = ui.load_runtime(p.workdir)
    for key, expected in (('repo_dir', p.repo_dir), ('workdir', p.workdir)):
        if os.path.realpath(str(rt.get(key))) != os.path.realpath(expected):
            raise ui.CellFailed('RUNTIME_MISMATCH', 'runtime.json có %s=%s, khác tham số %s.' % (key, rt.get(key), expected),
                                hint='Chạy lại cell 4 (bootstrap) cho repo/workdir hiện tại.')
    work = Path(str(rt.get('workdir')))
    start = ui.read_json(work / 'processes' / 'last_start.json')
    api_base = (((start or {}).get('api') or {}).get('base_url') or (rt.get('api') or {}).get('base_url')
                or ui.DEFAULT_API_BASE)

    def run_cli(script, *args, **kwargs):
        return ui.run_cli(rt, script, *args, **kwargs)

    def require_start():
        if not isinstance(start, dict) or not start.get('run_id') or not start.get('session_id'):
            raise ui.CellFailed('NO_SESSION', 'Chưa có session demo (TP_WORKDIR/processes/last_start.json).',
                                hint='Chạy cell 11-13 (prepare, start, wait-ready) trước.')
        return start

    return SimpleNamespace(p=p, ui=ui, rt=rt, repo=Path(str(rt.get('repo_dir'))), work=work,
                           core_python=rt['python']['core_python'], profile=rt.get('profile') or TP_PROFILE,
                           start=start, api_base=api_base, run_cli=run_cli, require_start=require_start)


_P = _params()
print('Tham số hợp lệ: ref=%s | repo=%s | workdir=%s' % (_P.git_ref, _P.repo_dir, _P.workdir))
print('Notebook benchmark: không chạy demo live và không mở tunnel.')
print('Python kernel: %s (core Python 3.11 nằm trong .venv của repo sau bootstrap)' % sys.version.split()[0])
for _msg in _P.notes:
    print('Lưu ý:', _msg)


In [ ]:
# [3/7] Clone/checkout đúng repository và ref. Chạy lại an toàn: tái sử dụng clone sạch đúng remote/ref;
# không bao giờ reset cứng hay xóa repo để che khác biệt, không đổi commit khi một demo session đang chạy.
# GITHUB_REPO_URL còn placeholder <OWNER>/<REPO> (chưa chạy scripts/configure_repository.py trước khi push):
# tái sử dụng checkout có sẵn của dự án tại TP_REPO_DIR, nếu không có thì hỏi URL MỘT lần. Không bao giờ tự đoán URL.
# Tùy chọn nâng cao (gán trong cell này hoặc cell 2 trước khi chạy):
#   REFRESH_REF = True   -> fetch và checkout lại GIT_REF dạng nhánh/tag (working tree phải sạch)
#   PRIVATE_REPO = True  -> repo private: token đọc từ Colab Secrets (GITHUB_TOKEN) hoặc getpass, chỉ nằm trong
#                           env của tiến trình git (askpass tạm 0700, xóa ngay), không lưu vào remote/notebook.
import getpass
import shutil
import subprocess
import tempfile
import urllib.parse

P = _params()
REFRESH_REF = bool(globals().get('REFRESH_REF', True))
PRIVATE_REPO = bool(globals().get('PRIVATE_REPO', False))
REPO = Path(P.repo_dir)
IS_SHA = re.fullmatch('[0-9a-f]{40}', P.git_ref) is not None
_SEG_RE = '[A-Za-z0-9_.-]{1,100}'


def _github_url(url):
    '''https://github.com/<owner>/<repo>(.git) -> https://github.com/<owner>/<repo>.git; None nếu sai/có credentials.'''
    url = (url or '').strip()
    try:
        parts = urllib.parse.urlsplit(url)
        port = parts.port
    except ValueError:
        return None
    if parts.scheme != 'https' or (parts.hostname or '').lower() != 'github.com':
        return None
    if parts.username or parts.password or '@' in parts.netloc or port is not None or parts.query or parts.fragment:
        return None
    segs = [s for s in parts.path.split('/') if s]
    if len(segs) != 2:
        return None
    owner = segs[0]
    name = segs[1][:-4] if segs[1].endswith('.git') else segs[1]
    for seg in (owner, name):
        if not re.fullmatch(_SEG_RE, seg) or seg.strip('.') == '':
            return None
    return 'https://github.com/%s/%s.git' % (owner, name)


def _is_project_checkout(path):
    return ((path / '.git').exists() and (path / 'notebooks' / 'colab_ui.py').is_file()
            and (path / 'scripts' / 'bootstrap_colab.sh').is_file())


REPO_URL = P.repo_url
if '<' in REPO_URL or '>' in REPO_URL:
    if _is_project_checkout(REPO):
        _origin = subprocess.run(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], stdout=subprocess.PIPE,
                                 stderr=subprocess.DEVNULL, text=True).stdout.strip()
        REPO_URL = _github_url(_origin)
        if REPO_URL is None:
            raise ParameterError('GITHUB_REPO_URL còn placeholder và origin của %s không phải URL '
                                 'https://github.com/<owner>/<repo> hợp lệ (không chứa thông tin đăng nhập). '
                                 'Sửa GITHUB_REPO_URL ở cell 2.' % REPO)
        print('GITHUB_REPO_URL còn placeholder: tái sử dụng checkout có sẵn tại', REPO, 'với origin', REPO_URL)
    else:
        print('GITHUB_REPO_URL còn placeholder <OWNER>/<REPO>. Cách khuyến nghị: chạy scripts/configure_repository.py '
              '--repo-url ... trước khi push để Run All không phải hỏi.')
        try:
            _answer = input('Nhập URL GitHub của repo (https://github.com/<owner>/<repo>.git): ')
        except Exception as exc:
            raise ParameterError('Không đọc được URL nhập vào (%s). Sửa GITHUB_REPO_URL ở cell 2 rồi chạy lại.'
                                 % type(exc).__name__)
        REPO_URL = _github_url(_answer)
        if REPO_URL is None:
            raise ParameterError('URL không hợp lệ: cần dạng https://github.com/<owner>/<repo>(.git), không chứa '
                                 'token/mật khẩu. Notebook không tự đoán URL; sửa GITHUB_REPO_URL ở cell 2 hoặc chạy lại cell này.')
    GITHUB_REPO_URL = REPO_URL  # các cell sau dùng URL đã xác nhận (chỉ trong kernel, không ghi vào file)
else:
    _url = urllib.parse.urlsplit(REPO_URL)
    if _url.scheme != 'https' or not _url.hostname or _url.username or _url.password or '@' in _url.netloc:
        raise ParameterError('GITHUB_REPO_URL phải là https và không chứa thông tin đăng nhập/token.')
    if _url.hostname.lower() != 'github.com':
        print('CẢNH BÁO: repo không nằm trên github.com:', _url.hostname)


def _norm_url(url):
    url = url.strip().rstrip('/')
    if url.endswith('.git'):
        url = url[:-4]
    parts = urllib.parse.urlsplit(url)
    return ('%s://%s%s' % (parts.scheme, parts.hostname or '', parts.path)).lower()


def _git(*args, env=None, timeout=900):
    cmd = ['git', '-c', 'credential.helper=', *args]
    r = subprocess.run(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, timeout=timeout)
    if r.returncode != 0:
        raise RuntimeError('git %s thất bại (exit %d): %s' % (' '.join(args[:4]), r.returncode, r.stderr.strip()[-2000:]))
    return r.stdout.strip()


def _git_ok(*args):
    return subprocess.run(['git', *args], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0


def _git_env():
    env = dict(os.environ)
    env['GIT_TERMINAL_PROMPT'] = '0'
    if not PRIVATE_REPO:
        return env, None
    token = None
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        token = None
    if not token:
        token = getpass.getpass('GitHub token (chỉ dùng cho tiến trình git này, không lưu): ')
    tmp = tempfile.mkdtemp(prefix='tp-askpass-')
    script = Path(tmp) / 'askpass.sh'
    script.write_text(chr(10).join(['#!/bin/sh', 'case $1 in', '  *sername*) echo x-access-token ;;',
                                    '  *) echo $TP_GIT_TOKEN ;;', 'esac', '']))
    os.chmod(script, 0o700)
    env['GIT_ASKPASS'] = str(script)
    env['TP_GIT_TOKEN'] = token
    return env, tmp


def _active_session():
    reg_path = Path(P.workdir) / 'processes' / 'registry.json'
    try:
        reg = json.loads(reg_path.read_text(encoding='utf-8'))
    except FileNotFoundError:
        return None
    except (OSError, ValueError):
        return 'không đọc được %s' % reg_path
    sid = reg.get('current_session')
    if not sid:
        return None
    state = None
    try:
        session_json = Path(P.workdir) / 'sessions' / str(sid) / 'session.json'
        state = json.loads(session_json.read_text(encoding='utf-8')).get('state')
    except (OSError, ValueError):
        state = None
    if state in ('STOPPED', 'FAILED'):
        return None
    return '%s (state=%s)' % (sid, state)


def _require_no_session(action):
    active = _active_session()
    if active:
        raise RuntimeError('SESSION_ACTIVE: không %s khi demo session %s đang chạy. Dừng session (cell 18 hoặc '
                           'demo.py stop) rồi chạy lại cell này.' % (action, active))


_env, _askpass_dir = _git_env()
action = 'REUSED'
try:
    if not REPO.exists():
        REPO.parent.mkdir(parents=True, exist_ok=True)
        if IS_SHA:
            _git('clone', '--no-checkout', REPO_URL, str(REPO), env=_env)
            _git('-C', str(REPO), 'checkout', '--detach', P.git_ref)
        else:
            _git('clone', '--branch', P.git_ref, '--single-branch', REPO_URL, str(REPO), env=_env)
        action = 'CLONED'
    else:
        if not (REPO / '.git').exists():
            raise RuntimeError('REPO_NOT_GIT: %s đã tồn tại nhưng không phải git repo. Notebook không xóa thư mục; '
                               'đổi TP_REPO_DIR hoặc tự di chuyển thư mục.' % REPO)
        origin = _git('-C', str(REPO), 'remote', 'get-url', 'origin')
        if _norm_url(origin) != _norm_url(REPO_URL):
            raise RuntimeError('REPO_REMOTE_MISMATCH: origin của %s khác GITHUB_REPO_URL=%s. Không ghi đè; '
                               'đổi TP_REPO_DIR hoặc sửa tham số.' % (REPO, REPO_URL))
        dirty = _git('-C', str(REPO), 'status', '--porcelain', '--untracked-files=no')
        if dirty:
            raise RuntimeError('REPO_DIRTY: working tree có thay đổi chưa commit:' + chr(10) + dirty + chr(10)
                               + 'Tự commit/stash/khôi phục; notebook không reset cứng hay ghi đè thay đổi.')
        head = _git('-C', str(REPO), 'rev-parse', 'HEAD')
        if IS_SHA:
            if head != P.git_ref:
                _require_no_session('đổi commit')
                if not _git_ok('-C', str(REPO), 'cat-file', '-e', P.git_ref + '^{commit}'):
                    _git('-C', str(REPO), 'fetch', 'origin', P.git_ref, env=_env)
                _git('-C', str(REPO), 'checkout', '--detach', P.git_ref)
                action = 'CHECKED_OUT'
        else:
            branch = _git('-C', str(REPO), 'rev-parse', '--abbrev-ref', 'HEAD')
            tags = _git('-C', str(REPO), 'tag', '--points-at', 'HEAD').split()
            on_ref = branch == P.git_ref or P.git_ref in tags
            if REFRESH_REF:
                _require_no_session('fetch/checkout lại ref')
                is_tag = bool(_git('-C', str(REPO), 'ls-remote', '--tags', 'origin', 'refs/tags/' + P.git_ref, env=_env))
                if is_tag:
                    _git('-C', str(REPO), 'fetch', 'origin', 'refs/tags/%s:refs/tags/%s' % (P.git_ref, P.git_ref), env=_env)
                    _git('-C', str(REPO), 'checkout', '--detach', 'refs/tags/' + P.git_ref)
                else:
                    _git('-C', str(REPO), 'fetch', 'origin', P.git_ref, env=_env)
                    if _git_ok('-C', str(REPO), 'show-ref', '--verify', '--quiet', 'refs/heads/' + P.git_ref):
                        _git('-C', str(REPO), 'checkout', P.git_ref)
                        _git('-C', str(REPO), 'merge', '--ff-only', 'FETCH_HEAD')
                    else:
                        _git('-C', str(REPO), 'checkout', '-b', P.git_ref, 'FETCH_HEAD')
                action = 'REFRESHED'
            elif not on_ref:
                raise RuntimeError('REF_MISMATCH: repo đang ở %s (%s), khác GIT_REF=%s. Đặt REFRESH_REF = True rồi '
                                   'chạy lại cell này (cần working tree sạch và không có session đang chạy).'
                                   % (branch, head[:12], P.git_ref))
finally:
    if _askpass_dir:
        shutil.rmtree(_askpass_dir, ignore_errors=True)
    _env = None

TP_REPO_COMMIT = _git('-C', str(REPO), 'rev-parse', 'HEAD')
_branch = _git('-C', str(REPO), 'rev-parse', '--abbrev-ref', 'HEAD')
_status = _git('-C', str(REPO), 'status', '--porcelain', '--untracked-files=normal')
_untracked = [line[3:] for line in _status.splitlines() if line.startswith('??')]
print('Repo        :', REPO, '(' + action + ')')
print('Remote      :', REPO_URL)
print('GIT_REF     :', P.git_ref, '| HEAD:', _branch)
print('Commit SHA  :', TP_REPO_COMMIT)
print('Working tree: sạch (không có thay đổi tracked)')
if _untracked:
    print('WARN untracked (uv.lock sinh trên Colab là bình thường):', ', '.join(_untracked[:20]))
if IS_SHA:
    print('Commit khớp GIT_REF.' if TP_REPO_COMMIT == P.git_ref else 'CẢNH BÁO: commit khác GIT_REF!')
else:
    print('Lưu ý: GIT_REF là nhánh/tag nên có thể trôi; notebook đang mở có thể thuộc commit khác commit vừa clone. '
          'Bản demo chính thức nên mở notebook và clone cùng một commit SHA hoặc release tag. '
          'Mọi report ghi SHA ở trên.')


In [ ]:
# [4/7] Bootstrap native runtime: uv + Python 3.11 (.venv), JDK 17, Kafka, Cassandra, Spark connector JARs,
# dashboard extra (Streamlit/PyDeck), Node.js đã ghim, bản build console React và cloudflared.
# ENABLE_CLOUDFLARE_TUNNEL=False thêm --no-tunnel: chỉ bỏ qua cloudflared, vẫn build UI. Lần đầu tải khoảng 1 GB
# và build pyspark (vài phút); chạy lại sẽ tái sử dụng cache đã kiểm checksum.
P = _params()
ui = _load_ui(P.repo_dir)
bootstrap_argv = ['/bin/bash', f'{TP_REPO_DIR}/scripts/bootstrap_colab.sh', '--workdir', TP_WORKDIR]
if not P.enable_cloudflare_tunnel:
    bootstrap_argv.append('--no-tunnel')
rc, _out, tail, secs = ui.stream_command(bootstrap_argv, cwd=P.repo_dir, env=ui.bootstrap_env(), timeout=1800,
                                         label='bootstrap_colab.sh')
ui.show_bootstrap_report(ui.read_json(Path(P.workdir) / 'runtime' / 'bootstrap_report.json'))
if rc != 0:
    ui.record_step(P.workdir, 'bootstrap', 'FAIL', exit_code=rc, no_tunnel=not P.enable_cloudflare_tunnel)
    raise ui.CellFailed('BOOTSTRAP_EXIT_%d' % rc, 'bootstrap_colab.sh thất bại.',
                        hint='Đọc bước FAIL trong bootstrap report và log phía trên; sửa nguyên nhân rồi chạy lại '
                             '(cache đã tải được giữ lại).',
                        log_excerpt=chr(10).join(tail[-60:]), exit_code=rc)
rt = ui.load_runtime(P.workdir)
_py, _java, _spark = rt.get('python') or {}, rt.get('java') or {}, rt.get('spark') or {}
ui.show_kv({
    'core_python': _py.get('core_python'), 'python': _py.get('version'), 'lock_origin': _py.get('lock_origin'),
    'java': '%s %s' % (_java.get('vendor'), _java.get('version')), 'spark': _spark.get('version'),
    'connector_jars': len(_spark.get('connector_jars') or []), 'kafka_home': (rt.get('kafka') or {}).get('home'),
    'cassandra_home': (rt.get('cassandra') or {}).get('home'), 'repo_commit': rt.get('repo_commit'),
    'api': (rt.get('api') or {}).get('base_url'),
    'cloudflared': 'được cài' if P.enable_cloudflare_tunnel else 'bỏ qua (--no-tunnel)',
}, title='runtime.json')
if _py.get('lock_origin') == 'generated_on_colab':
    print('uv.lock được sinh trên Colab (lock_origin=generated_on_colab); file này có trong evidence ZIP để commit lại.')
ui.record_step(P.workdir, 'bootstrap', 'PASS', seconds=round(secs, 1), lock_origin=_py.get('lock_origin'),
               no_tunnel=not P.enable_cloudflare_tunnel)


In [ ]:
ctx = _ctx()  # [5/7] Start Kafka (KRaft) + Cassandra native, rồi init topics + CQL migrations (idempotent).
# Chạy lại an toàn: supervisor tái sử dụng broker/node đang chạy (một broker, một node).
infra_start = ctx.run_cli('infra.py', 'start', '--profile', ctx.profile, timeout=900, label='infra start',
                          show=lambda d: ctx.ui.show_cli_result(d, 'infra start (Kafka + Cassandra)', ('reused',)))
infra_init = ctx.run_cli('infra.py', 'init', '--profile', ctx.profile, timeout=900, label='infra init',
                         show=lambda d: ctx.ui.show_cli_result(d, 'infra init (topics + CQL)', ('cluster_id', 'release_version')))
ctx.ui.record_step(ctx.work, 'infra', 'PASS')


In [ ]:
ctx = _ctx()  # [6/7] Tắt API rồi chạy toàn bộ scenario. Một lần lặp mỗi scenario. Không giới hạn 300 giây.

# Sửa ba số này nếu muốn đo lâu hơn hoặc lặp lại. p95 cần ít nhất 20 mẫu trong cửa sổ đo.
# Warm-up 180 s cho lần compile Spark của process benchmark, trước khi bắt đầu đo.
REPETITIONS = 1
WARMUP_SECONDS = 180
MEASURE_SECONDS = 180

for args in (["expose", "stop", "--target", "all"], ["stop"]):
    try:
        ctx.run_cli("demo.py", *args, timeout=180, check=False, label="demo.py " + " ".join(args))
    except Exception as exc:
        print("Bỏ qua bước dừng (%s): %s" % (type(exc).__name__, exc))

manifest = (ctx.p.manifest or "").strip()
if not manifest:
    found = sorted(Path(ctx.work).glob("data/raw/*/recording_manifest.json"), key=lambda p: p.stat().st_mtime)
    manifest = str(found[-1]) if found else ""

scenarios = [
    "stress_100", "stress_500", "stress_1000",
    "recovery_partial_write", "recovery_after_cas",
    "recovery_cassandra_down", "recovery_collector_outbox",
]
if manifest:
    scenarios = ["replay_1x", "replay_10x", "replay_50x"] + scenarios
    print("Recording:", manifest)
else:
    print("Không có recording live. Bỏ replay_1x, replay_10x, replay_50x.")
print("Scenario:", ", ".join(scenarios))

argv = [
    ctx.core_python, str(ctx.repo / "benchmarks" / "run_benchmark.py"),
    "--scenario", ",".join(scenarios),
    "--profile", ctx.profile,
    "--workdir", str(ctx.work),
    "--repetitions", str(REPETITIONS),
    "--warmup", str(WARMUP_SECONDS),
    "--measure", str(MEASURE_SECONDS),
]
if manifest:
    argv += ["--recording-manifest", manifest]

try:
    rc, out, tail, secs = ctx.ui.stream_command(
        argv, cwd=ctx.repo, timeout=21600, label="benchmark toàn bộ", capture_stdout=True,
    )
except ctx.ui.CellFailed as exc:
    print("Benchmark bị cắt:", exc.code, exc.message)
    rc, out = 1, ""
else:
    print("Benchmark thoát mã", rc, "sau %.0f s" % secs)

doc = ctx.ui.parse_json_document(out) or {}
print("bench_id:", doc.get("bench_id"))
print("thư mục:", doc.get("out_dir"))
print("trạng thái:", doc.get("statuses"))
for row in doc.get("results") or []:
    print("-", row.get("scenario"), "lần", row.get("repetition"), row.get("status"),
          (row.get("notes") or [""])[0][:240])
if rc != 0 and not doc:
    print("Không có JSON kết quả. Xem log phía trên.")


In [ ]:
ctx = _ctx()  # [7/7] Zip mọi lần đo trong workdir và tải về máy.

import shutil

report_root = Path(ctx.work) / "reports"
bench_dir = report_root / "benchmarks"
if not bench_dir.is_dir() or not any(bench_dir.iterdir()):
    raise ctx.ui.CellFailed(
        "NO_BENCHMARK_OUTPUT",
        "Chưa có thư mục reports/benchmarks.",
        hint="Chạy cell benchmark trước. Zip chỉ có sau khi harness ghi report.",
    )
archive = Path("/content/benchmarks")
zip_path = shutil.make_archive(str(archive), "zip", root_dir=str(report_root), base_dir="benchmarks")
print("Đã nén", zip_path, "(%d byte)" % Path(zip_path).stat().st_size)
try:
    from google.colab import files
    files.download(zip_path)
    print("Trình duyệt sẽ hỏi chỗ lưu benchmarks.zip.")
except Exception as exc:
    print("Không mở được hộp tải (%s). File vẫn nằm tại %s" % (exc, zip_path))
